In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
import joblib

# --- 1. ЗАВАНТАЖЕННЯ ДАНИХ ---
# Додаємо всі необхідні колонки
cols_to_use = ['loan_status', 'annual_inc', 'loan_amnt', 'term', 'fico_range_low', 'dti']

print("⏳ Завантаження даних...")
# Читаємо 200,000 рядків (або більше, якщо дозволяє пам'ять)
df = pd.read_csv("data/accepted_2007_to_2018Q4.csv", usecols=cols_to_use, nrows=200000)
print(f"✅ Завантажено {len(df)} рядків.")

# --- 2. ОЧИЩЕННЯ ТА ПІДГОТОВКА (Feature Engineering) ---
print("🧹 Починаємо очищення та створення нових ознак...")

# 2.1. Фільтрація статусів
df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
df['target'] = df['loan_status'].apply(lambda x: 1 if x == 'Fully Paid' else 0)

# 2.2. Обробка терміну (Term)
df['term'] = df['term'].str.extract('(\d+)').astype(float).fillna(36)

# 2.3. Заповнення пропусків
df['annual_inc'] = df['annual_inc'].fillna(df['annual_inc'].mean())
df['fico_range_low'] = df['fico_range_low'].fillna(df['fico_range_low'].mean())
df = df.dropna(subset=['dti'])

# --- 2.4. ВАЖЛИВІ ФІЛЬТРИ (ВИДАЛЯЄМО СМІТТЯ) ---
# Видаляємо нереалістичні дані, які плутають модель
df = df[df['annual_inc'] > 1000]   # Дохід менше 1000/рік — це помилка
df = df[df['annual_inc'] < 300000] # Обрізаємо супер-багатіїв (викиди)
df = df[df['dti'] <= 100]          # DTI більше 100% не буває в нормі

# --- 2.5. СТВОРЕННЯ НОВОЇ ОЗНАКИ (GAME CHANGER) ---
# Відношення суми кредиту до річного доходу.
# Якщо це число велике (наприклад > 0.5), ризик величезний.
df['loan_to_income'] = df['loan_amnt'] / df['annual_inc']

print(f"📊 Даних для навчання після чистки: {len(df)} рядків.")

# --- 3. НАВЧАННЯ МОДЕЛІ ---
# Тепер у нас 6 ознак
feature_cols = ['annual_inc', 'loan_amnt', 'term', 'fico_range_low', 'dti', 'loan_to_income']
X = df[feature_cols]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("🧠 Тренування Random Forest (з новими ознаками)...")
model = RandomForestClassifier(n_estimators=100, max_depth=12, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

# --- 4. ПЕРЕВІРКА ЯКОСТІ ---
y_pred = model.predict(X_test)
prob_test = model.predict_proba(X_test)[:, 1]

print("\n--- РЕЗУЛЬТАТИ ---")
print(classification_report(y_test, y_pred))
print(f"🚀 ROC-AUC Score: {roc_auc_score(y_test, prob_test):.3f}")

# --- 5. ЗБЕРЕЖЕННЯ ---
# Зберігаємо модель
joblib.dump(model, "credit_model.pkl")
print("💾 Файл 'credit_model.pkl' оновлено успішно!")

<>:27: SyntaxWarning: invalid escape sequence '\d'
<>:27: SyntaxWarning: invalid escape sequence '\d'
C:\Users\n1107\AppData\Local\Temp\ipykernel_12756\3315207657.py:27: SyntaxWarning: invalid escape sequence '\d'
  df['term'] = df['term'].str.extract('(\d+)').astype(float).fillna(36)


⏳ Завантаження даних...
✅ Завантажено 200000 рядків.
🧹 Починаємо очищення та створення нових ознак...
📊 Даних для навчання після чистки: 174711 рядків.
🧠 Тренування Random Forest (з новими ознаками)...

--- РЕЗУЛЬТАТИ ---
              precision    recall  f1-score   support

           0       0.36      0.53      0.43      7034
           1       0.87      0.76      0.81     27909

    accuracy                           0.72     34943
   macro avg       0.61      0.65      0.62     34943
weighted avg       0.76      0.72      0.73     34943

🚀 ROC-AUC Score: 0.705
💾 Файл 'credit_model.pkl' оновлено успішно!
